# Multi-Metric Performance Evaluation

This notebook computes comprehensive evaluation metrics for our three primary models:
1. **CheXNet Baseline (224px)**
2. **Novel Hybrid (224px)**
3. **Novel Hybrid (384px)**

### Metrics Computed:
- **Macro AUC**
- **Per-class AUC**
- **Macro F1 Score**
- **Precision & Recall**

In [ ]:
import os
import cv2
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import timm
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from PIL import Image
from tqdm.auto import tqdm

# --- CONFIG ---
DATA_DIR = '../data/images'
METADATA_PATH = '../data/metadata_filtered.csv'
TARGET_CLASSES = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 
    'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 
    'No Finding', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax'
]

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Model Definitions

In [ ]:
def get_chexnet_model():
    model = models.densenet121(weights=None)
    model.classifier = nn.Linear(model.classifier.in_features, len(TARGET_CLASSES))
    return model

class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=7, padding=3)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        res = torch.cat([avg_out, max_out], dim=1)
        res = self.conv(res)
        return x * self.sigmoid(res)

class HybridCheXNet(nn.Module):
    def __init__(self, img_size):
        super().__init__()
        cnn = models.densenet121(weights=None)
        self.cnn_features = cnn.features
        self.cnn_pool = nn.AdaptiveAvgPool2d(1)
        self.vit = timm.create_model('swin_tiny_patch4_window7_224', pretrained=False, num_classes=0, img_size=img_size)
        self.spatial_att = SpatialAttention()
        self.classifier = nn.Sequential(nn.Linear(1024 + 768, 512), nn.ReLU())
        self.final_fc = nn.Linear(512, len(TARGET_CLASSES))
    def forward(self, x):
        c_feat = self.cnn_features(x)
        c_feat = self.spatial_att(c_feat)
        c_feat = self.cnn_pool(c_feat).view(x.size(0), -1) 
        v_feat = self.vit(x)
        combined = torch.cat([c_feat, v_feat], dim=1)
        out = self.classifier(combined)
        return self.final_fc(out)

## 2. Data Pipeline

In [ ]:
class MetricsDataset(Dataset):
    def __init__(self, dataframe, root_dir, img_size, transform=None):
        self.dataframe = dataframe
        self.root_dir = root_dir
        self.img_size = img_size
        self.transform = transform
        self.clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        try:
            img_name = self.dataframe.iloc[idx]['Image Index']
            img_path = os.path.join(self.root_dir, img_name)
            image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if image is None: return torch.zeros((3, self.img_size, self.img_size)), torch.zeros(len(TARGET_CLASSES))
            image = cv2.resize(image, (self.img_size, self.img_size), interpolation=cv2.INTER_AREA)
            image = self.clahe.apply(image)
            image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
            image = Image.fromarray(image)
            label = torch.tensor(self.dataframe.iloc[idx]['label_vec'], dtype=torch.float32)
            if self.transform: image = self.transform(image)
            return image, label
        except: return torch.zeros((3, self.img_size, self.img_size)), torch.zeros(len(TARGET_CLASSES))

# Data Loading
df = pd.read_csv(METADATA_PATH)
def encode(l): return [1 if c in str(l).split('|') else 0 for c in TARGET_CLASSES]
df['label_vec'] = df['Finding Labels'].apply(encode)

# Consistent Sampling & Split
df = df.sample(n=3000, random_state=42).reset_index(drop=True) # 3,000 for standard benchmark
_, val_df = train_test_split(df, test_size=0.33, random_state=42) # 1,000 images validation

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## 3. Metric Computation Engine

In [ ]:
def compute_metrics(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(loader):
            out = torch.sigmoid(model(imgs.to(device)))
            all_preds.append(out.cpu().numpy())
            all_labels.append(labels.numpy())
    
    preds = np.vstack(all_preds)
    labels = np.vstack(all_labels)
    
    # Thresholding for binary metrics (F1, Precision, Recall)
    binary_preds = (preds > 0.5).astype(int)
    
    # Macro Metrics
    macro_auc = roc_auc_score(labels, preds, average='macro')
    macro_f1 = f1_score(labels, binary_preds, average='macro', zero_division=0)
    precision = precision_score(labels, binary_preds, average='macro', zero_division=0)
    recall = recall_score(labels, binary_preds, average='macro', zero_division=0)
    
    # Per-Class AUC
    per_class_auc = {}
    for i, name in enumerate(TARGET_CLASSES):
        try: per_class_auc[name] = roc_auc_score(labels[:, i], preds[:, i])
        except: per_class_auc[name] = 0.5
        
    return {
        "Macro AUC": macro_auc,
        "Macro F1": macro_f1,
        "Precision": precision,
        "Recall": recall,
        "Per-Class AUC": per_class_auc
    }

## 4. Run Evaluation

In [ ]:
results = {}

# 1. CheXNet
model = get_chexnet_model().to(device)
model.load_state_dict(torch.load('../chexnet_baseline/chexnet_best.pth', map_location=device))
loader = DataLoader(MetricsDataset(val_df, DATA_DIR, 224, transform), batch_size=16)
results['CheXNet 224'] = compute_metrics(model, loader)

# 2. Novel Hybrid 224
model = HybridCheXNet(224).to(device)
model.load_state_dict(torch.load('../novel_pipeline_224/novel_full_best.pth', map_location=device))
results['Novel Hybrid 224'] = compute_metrics(model, loader)

# 3. Novel Hybrid 384
model = HybridCheXNet(384).to(device)
model.load_state_dict(torch.load('../novel_pipeline_384_highres/novel_full_384_best.pth', map_location=device))
loader_384 = DataLoader(MetricsDataset(val_df, DATA_DIR, 384, transform), batch_size=16)
results['Novel Hybrid 384'] = compute_metrics(model, loader_384)

## 5. Summary Results

In [ ]:
summary_data = []
for model_name, metrics in results.items():
    summary_data.append({
        "Model": model_name,
        "Macro AUC": metrics["Macro AUC"],
        "Macro F1": metrics["Macro F1"],
        "Precision": metrics["Precision"],
        "Recall": metrics["Recall"]
    })

summary_df = pd.DataFrame(summary_data)
display(summary_df)

# Detailed Pathology AUC Table
pathology_data = {"Pathology": TARGET_CLASSES}
for model_name, metrics in results.items():
    pathology_data[model_name] = [metrics["Per-Class AUC"][c] for c in TARGET_CLASSES]

pathology_df = pd.DataFrame(pathology_data)
display(pathology_df)